# Моделирование динамики информационных событий через Hawkes-процесс

Этот ноутбук реализует **оба варианта** для диплома:

1. **Вариант 1: синтетика**  
   Генерируем каскады через `tick.hawkes.SimuHawkesExpKernels`, затем проверяем, насколько хорошо модель восстанавливает параметры `mu`, `alpha`, `decay`.

2. **Вариант 2: реальный датасет**  
   Для каждого информационного события (кластера) берём времена публикаций, обучаем одномерный Hawkes-процесс и строим прогноз по первым `N` сообщениям.

## Что именно оцениваем

- `mu` — фоновая интенсивность.
- `alpha` — сила самовозбуждения.
- `decay` — скорость затухания влияния прошлых сообщений.

## Метрики

- `RMSE` числа сообщений в ближайший час.
- `RMSE` итогового размера каскада.
- Корреляция между `alpha` и фактическим размером каскада.

## Важное замечание по окружению

У `tick` есть важный нюанс по установке: релиз на PyPI устарел, поэтому для стабильного запуска лучше использовать отдельное окружение `Python 3.11` или `3.12` и ставить актуальную версию из GitHub:

```bash
conda create -n hawkes-diploma python=3.12 -y
conda activate hawkes-diploma
pip install -U pip setuptools wheel
unset CC CXX
pip install -r requirements.txt
jupyter notebook
```

Для Ubuntu перед этим обычно нужно поставить системный toolchain:

```bash
sudo apt-get update
sudo apt-get install -y build-essential cmake ninja-build git python3-dev python3-venv
```

Если установка всё равно падает, сначала проверьте системные build-tools: C++ compiler и `cmake`. Отдельно проверьте переменные окружения `CC` и `CXX`: если они указывают на несуществующий компилятор, их нужно сбросить через `unset CC CXX`. Старый PyPI-релиз `tick` может требовать `swig`, но актуальная версия из GitHub использует другой сборочный путь.

## Архитектура решения

### 1. Подготовка данных

На вход нужен датасет с минимум двумя полями:

- `cluster_id` — идентификатор информационного события.
- `published_at` — время публикации сообщения.

Если в датасете **нет готовых кластеров**, Hawkes-модель обучать рано: сначала нужен отдельный этап кластеризации новостей по событию.

### 2. Построение каскадов

Для каждого `cluster_id`:

- сортируем публикации по времени;
- переводим время в секунды;
- сдвигаем начало каскада в `t = 0`.

### 3. Обучение модели

Для одного кластера используем одномерный Hawkes-процесс с экспоненциальным ядром:

$$
\lambda(t) = \mu + \sum_{t_i < t} \alpha \cdot \beta \cdot e^{-\beta (t - t_i)}
$$

где:

- `mu` — фон,
- `alpha` — сила возбуждения,
- `beta = decay` — скорость затухания.

### 4. Оценка `decay`

Практический нюанс: в `tick` класс `HawkesExpKern` обычно обучает `mu` и `alpha` **при фиксированном `decay`**. Поэтому в ноутбуке `decay` выбирается через **поиск по сетке**: пробуем несколько значений и берём лучшее по логарифму правдоподобия.

### 5. Прогноз по первым `N` сообщениям

Берём только префикс каскада:

- первые `N` сообщений;
- обучаем `mu`, `alpha`, `decay`;
- прогнозируем:
  - сколько сообщений придёт в ближайший час;
  - каков будет итоговый размер каскада в окне наблюдения.

### 6. Оценка качества

По набору каскадов считаем:

- `RMSE(next_hour_count)`
- `RMSE(final_size)`
- `corr(alpha, final_size)`

In [ ]:
from __future__ import annotations

import math
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from tick.hawkes import HawkesExpKern, SimuHawkesExpKernels

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_rmse(y_true, y_pred):
    if len(y_true) == 0:
        return np.nan
    return rmse(y_true, y_pred)


def safe_corr(x, y):
    if len(x) < 2 or len(y) < 2:
        return np.nan
    return float(pd.Series(x).corr(pd.Series(y)))


def as_univariate_realization(timestamps):
    ts = np.asarray(timestamps, dtype=float)
    ts = np.sort(ts)
    return [[ts]]


def fit_hawkes_exp_grid(timestamps, decay_grid, max_iter=300, tol=1e-5):
    """
    Подбирает decay по сетке и обучает mu/alpha через tick.HawkesExpKern.
    Возвращает лучшую модель по score (лог-правдоподобие на обучающей выборке).
    """
    ts = np.asarray(timestamps, dtype=float)
    ts = np.sort(ts)

    if len(ts) < 2:
        raise ValueError("Для оценки Hawkes-процесса нужно минимум 2 события.")

    events = as_univariate_realization(ts)
    best = None

    for decay in decay_grid:
        decay = float(decay)
        learner = HawkesExpKern(
            decays=np.array([[decay]], dtype=float),
            penalty="none",
            solver="agd",
            max_iter=max_iter,
            tol=tol,
        )
        learner.fit(events)
        score = float(learner.score(events))
        candidate = {
            "model": learner,
            "score": score,
            "mu": float(learner.baseline[0]),
            "alpha": float(learner.adjacency[0, 0]),
            "decay": decay,
        }
        if best is None or candidate["score"] > best["score"]:
            best = candidate

    return best


def excitation_state(history, decay, t_ref=None):
    history = np.asarray(history, dtype=float)
    if len(history) == 0:
        return 0.0
    if t_ref is None:
        t_ref = float(history[-1])
    return float(np.exp(-decay * (t_ref - history[history <= t_ref])).sum())


def hawkes_intensity(mu, alpha, decay, history, t):
    history = np.asarray(history, dtype=float)
    past = history[history < t]
    if len(past) == 0:
        return float(mu)
    return float(mu + alpha * decay * np.exp(-decay * (t - past)).sum())


def intensity_curve(mu, alpha, decay, history, grid):
    return np.array([hawkes_intensity(mu, alpha, decay, history, t) for t in grid], dtype=float)


def simulate_future_counts(mu, alpha, decay, history, horizon_sec, n_simulations=200, seed=42):
    """
    Монте-Карло прогноз будущего числа событий на горизонте horizon_sec.
    Реализован через thinning для одномерного Hawkes с экспоненциальным ядром.
    """
    history = np.asarray(history, dtype=float)
    if len(history) == 0:
        raise ValueError("История событий не может быть пустой.")

    t0 = float(history[-1])
    base_state = excitation_state(history, decay, t_ref=t0)
    rng_local = np.random.default_rng(seed)
    counts = []

    for _ in range(n_simulations):
        t = 0.0
        g = base_state
        count = 0
        current_lambda = mu + alpha * decay * g

        while t < horizon_sec:
            lambda_bar = max(current_lambda, mu + 1e-12)
            wait = -math.log(rng_local.uniform()) / lambda_bar
            t += wait
            if t > horizon_sec:
                break

            g = g * math.exp(-decay * wait)
            lambda_t = mu + alpha * decay * g

            if rng_local.uniform() <= lambda_t / lambda_bar:
                count += 1
                g += 1.0
                current_lambda = mu + alpha * decay * g
            else:
                current_lambda = lambda_t

        counts.append(count)

    return np.asarray(counts, dtype=int)


def actual_future_count(full_timestamps, prefix_size, horizon_sec):
    ts = np.asarray(full_timestamps, dtype=float)
    cutoff = ts[prefix_size - 1]
    return int(((ts > cutoff) & (ts <= cutoff + horizon_sec)).sum())


def evaluate_prefix_forecast(full_timestamps, prefix_size, decay_grid, hour_horizon=3600, n_simulations=200, seed=42):
    ts = np.asarray(full_timestamps, dtype=float)
    ts = np.sort(ts)
    if len(ts) <= prefix_size:
        raise ValueError("Каскад слишком короткий для выбранного prefix_size.")

    prefix_abs = ts[:prefix_size]
    prefix_rel = prefix_abs - prefix_abs[0]

    fit = fit_hawkes_exp_grid(prefix_rel, decay_grid)
    remaining_observed_horizon = float(ts[-1] - prefix_abs[-1])

    future_hour_samples = simulate_future_counts(
        mu=fit["mu"],
        alpha=fit["alpha"],
        decay=fit["decay"],
        history=prefix_rel,
        horizon_sec=hour_horizon,
        n_simulations=n_simulations,
        seed=seed,
    )

    future_total_samples = simulate_future_counts(
        mu=fit["mu"],
        alpha=fit["alpha"],
        decay=fit["decay"],
        history=prefix_rel,
        horizon_sec=max(remaining_observed_horizon, 1.0),
        n_simulations=n_simulations,
        seed=seed + 1,
    )

    return {
        "prefix_size": prefix_size,
        "mu": fit["mu"],
        "alpha": fit["alpha"],
        "decay": fit["decay"],
        "score": fit["score"],
        "actual_next_hour": actual_future_count(ts, prefix_size, hour_horizon),
        "pred_next_hour": float(future_hour_samples.mean()),
        "actual_total": int(len(ts)),
        "pred_total": float(prefix_size + future_total_samples.mean()),
        "actual_future_total": int(len(ts) - prefix_size),
        "pred_future_total": float(future_total_samples.mean()),
    }


def simulate_synthetic_cascade(mu, alpha, decay, end_time_sec, seed=None):
    simulator = SimuHawkesExpKernels(
        adjacency=np.array([[alpha]], dtype=float),
        decays=np.array([[decay]], dtype=float),
        baseline=np.array([mu], dtype=float),
        end_time=float(end_time_sec),
        seed=seed,
        verbose=False,
    )
    simulator.simulate()
    return np.asarray(simulator.timestamps[0], dtype=float)


def generate_synthetic_dataset(
    n_cascades=25,
    end_time_sec=24 * 3600,
    min_events=30,
    max_attempts=30,
    seed=42,
):
    rng_local = np.random.default_rng(seed)
    cascades = []

    for cascade_id in range(n_cascades):
        for attempt in range(max_attempts):
            mu = rng_local.uniform(2e-4, 10e-4)
            alpha = rng_local.uniform(0.15, 0.85)
            decay = rng_local.uniform(1 / 14400, 1 / 1200)
            ts = simulate_synthetic_cascade(mu, alpha, decay, end_time_sec=end_time_sec, seed=seed + cascade_id * 100 + attempt)
            if len(ts) >= min_events:
                cascades.append({
                    "cascade_id": cascade_id,
                    "timestamps": ts,
                    "mu_true": mu,
                    "alpha_true": alpha,
                    "decay_true": decay,
                    "n_events": int(len(ts)),
                    "duration_sec": float(ts[-1] - ts[0]) if len(ts) > 1 else 0.0,
                })
                break
        else:
            warnings.warn(f"Не удалось сгенерировать каскад {cascade_id} с минимум {min_events} событиями.")

    return cascades


def load_dataset(path):
    path = str(path)
    suffix = Path(path).suffix.lower()

    if path.startswith("hf://"):
        return pd.read_json(path, lines=True)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=True)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in {".parquet", ".pq"}:
        return pd.read_parquet(path)

    raise ValueError(f"Неподдерживаемый формат файла: {path}")


def prepare_cascades(df, cluster_col, time_col, min_posts=10, max_cascades=None):
    work = df[[cluster_col, time_col]].copy()
    work[time_col] = pd.to_datetime(work[time_col], utc=True, errors="coerce")
    work = work.dropna(subset=[cluster_col, time_col]).sort_values([cluster_col, time_col])

    cascades = []
    for cluster_id, group in work.groupby(cluster_col):
        timestamps_sec = group[time_col].astype("int64").to_numpy() / 1e9
        if len(timestamps_sec) < min_posts:
            continue
        timestamps_rel = timestamps_sec - timestamps_sec[0]
        cascades.append({
            "cluster_id": cluster_id,
            "timestamps": timestamps_rel,
            "n_posts": int(len(timestamps_rel)),
            "duration_sec": float(timestamps_rel[-1]) if len(timestamps_rel) else 0.0,
        })

    cascades = sorted(cascades, key=lambda x: x["n_posts"], reverse=True)
    if max_cascades is not None:
        cascades = cascades[:max_cascades]
    return cascades


def evaluate_cascades(cascades, prefix_size, decay_grid, hour_horizon=3600, n_simulations=200, seed=42):
    rows = []
    for idx, cascade in enumerate(cascades):
        ts = np.asarray(cascade["timestamps"], dtype=float)
        if len(ts) <= prefix_size:
            continue
        try:
            metrics = evaluate_prefix_forecast(
                full_timestamps=ts,
                prefix_size=prefix_size,
                decay_grid=decay_grid,
                hour_horizon=hour_horizon,
                n_simulations=n_simulations,
                seed=seed + idx,
            )
            rows.append({**cascade, **metrics})
        except Exception as exc:
            warnings.warn(f"Каскад {cascade.get('cluster_id', cascade.get('cascade_id', idx))} пропущен: {exc}")

    return pd.DataFrame(rows)

## Вариант 1. Синтетика

Цель этого блока: показать, что на контролируемых данных модель восстанавливает параметры и даёт разумный прогноз.

Логика:

1. Генерируем набор искусственных каскадов.
2. Для каждого каскада знаем истинные `mu`, `alpha`, `decay`.
3. Обучаем Hawkes-модель обратно по сгенерированным временам событий.
4. Сравниваем истинные и восстановленные параметры.
5. Проверяем качество прогноза по первым `N` событиям.

In [ ]:
SYNTHETIC_END_TIME_SEC = 24 * 3600
SYNTHETIC_PREFIX_SIZE = 20
SYNTHETIC_HOUR_HORIZON = 3600
DECAY_GRID = np.geomspace(1 / 21600, 1 / 600, 16)

synthetic_cascades = generate_synthetic_dataset(
    n_cascades=30,
    end_time_sec=SYNTHETIC_END_TIME_SEC,
    min_events=35,
    seed=RANDOM_SEED,
)

len(synthetic_cascades), synthetic_cascades[0]["n_events"]

In [ ]:
synthetic_recovery_rows = []
for cascade in synthetic_cascades:
    fit = fit_hawkes_exp_grid(cascade["timestamps"], DECAY_GRID)
    synthetic_recovery_rows.append({
        "cascade_id": cascade["cascade_id"],
        "n_events": cascade["n_events"],
        "mu_true": cascade["mu_true"],
        "mu_hat": fit["mu"],
        "alpha_true": cascade["alpha_true"],
        "alpha_hat": fit["alpha"],
        "decay_true": cascade["decay_true"],
        "decay_hat": fit["decay"],
        "score": fit["score"],
    })

synthetic_recovery_df = pd.DataFrame(synthetic_recovery_rows)

summary_recovery = pd.DataFrame({
    "metric": ["RMSE(mu)", "RMSE(alpha)", "RMSE(decay)"],
    "value": [
        rmse(synthetic_recovery_df["mu_true"], synthetic_recovery_df["mu_hat"]),
        rmse(synthetic_recovery_df["alpha_true"], synthetic_recovery_df["alpha_hat"]),
        rmse(synthetic_recovery_df["decay_true"], synthetic_recovery_df["decay_hat"]),
    ],
})

summary_recovery

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, true_col, pred_col, title in [
    (axes[0], "mu_true", "mu_hat", "mu: истинное vs оценка"),
    (axes[1], "alpha_true", "alpha_hat", "alpha: истинное vs оценка"),
    (axes[2], "decay_true", "decay_hat", "decay: истинное vs оценка"),
]:
    ax.scatter(synthetic_recovery_df[true_col], synthetic_recovery_df[pred_col], alpha=0.8)
    line_min = min(synthetic_recovery_df[true_col].min(), synthetic_recovery_df[pred_col].min())
    line_max = max(synthetic_recovery_df[true_col].max(), synthetic_recovery_df[pred_col].max())
    ax.plot([line_min, line_max], [line_min, line_max], linestyle="--", color="black")
    ax.set_title(title)
    ax.set_xlabel("Истинное значение")
    ax.set_ylabel("Оценка модели")

plt.tight_layout()
plt.show()

In [ ]:
synthetic_eval_df = evaluate_cascades(
    cascades=synthetic_cascades,
    prefix_size=SYNTHETIC_PREFIX_SIZE,
    decay_grid=DECAY_GRID,
    hour_horizon=SYNTHETIC_HOUR_HORIZON,
    n_simulations=250,
    seed=RANDOM_SEED,
)

synthetic_metrics = pd.DataFrame({
    "metric": [
        "RMSE(next_hour_count)",
        "RMSE(final_size)",
        "corr(alpha, final_size)",
    ],
    "value": [
        rmse(synthetic_eval_df["actual_next_hour"], synthetic_eval_df["pred_next_hour"]),
        rmse(synthetic_eval_df["actual_total"], synthetic_eval_df["pred_total"]),
        float(synthetic_eval_df["alpha"].corr(synthetic_eval_df["actual_total"])),
    ],
})

synthetic_metrics

In [ ]:
example = synthetic_cascades[np.argmax([c["n_events"] for c in synthetic_cascades])]
example_ts = example["timestamps"]
example_prefix = example_ts[:SYNTHETIC_PREFIX_SIZE]
example_fit = fit_hawkes_exp_grid(example_prefix, DECAY_GRID)

grid = np.linspace(0, min(example_ts[-1], example_prefix[-1] + 6 * 3600), 500)
intensity_vals = intensity_curve(
    mu=example_fit["mu"],
    alpha=example_fit["alpha"],
    decay=example_fit["decay"],
    history=example_prefix,
    grid=grid,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].step(example_ts / 3600, np.arange(1, len(example_ts) + 1), where="post")
axes[0].axvline(example_prefix[-1] / 3600, color="red", linestyle="--", label=f"prefix N={SYNTHETIC_PREFIX_SIZE}")
axes[0].set_title("Синтетический каскад: накопленное число событий")
axes[0].set_xlabel("Время, часы")
axes[0].set_ylabel("Число событий")
axes[0].legend()

axes[1].plot(grid / 3600, intensity_vals, color="darkgreen")
axes[1].vlines(example_prefix / 3600, ymin=0, ymax=intensity_vals.max(), color="gray", alpha=0.2)
axes[1].set_title("Оценённая интенсивность по первым N событиям")
axes[1].set_xlabel("Время, часы")
axes[1].set_ylabel("Интенсивность")

plt.tight_layout()
plt.show()

## Вариант 2. Реальный датасет

Здесь предполагается, что у вас уже есть таблица с публикациями и привязкой каждой публикации к событию (`cluster_id`).

### Минимально необходимые поля

- `cluster_id` — номер события / кластера.
- `published_at` — дата и время публикации.

### Если у вас другой нейминг колонок

Просто поменяйте значения в конфиге ниже.

### Если у вас ещё нет кластеров

Это отдельная задача. Для диплома можно честно написать так:

- сначала новости группируются в информационные события внешним методом;
- затем для каждого события моделируется динамика публикаций через Hawkes-процесс.

In [ ]:
# Укажите здесь свой путь к реальным данным.
# Если хотите читать из HuggingFace через pandas, можно использовать и remote-path,
# но для воспроизводимости диплома лучше иметь локальный файл.
DATA_PATH = "data/russian_news_clusters.jsonl"

# Примеры альтернатив:
# DATA_PATH = "hf://datasets/ScoutieAutoML/russian-news-telegram-dataset/scoutieRussianNewsTelegramDataset.json"
# DATA_PATH = "data/news_clusters.csv"

CLUSTER_COL = "cluster_id"
TIME_COL = "published_at"
MIN_POSTS_PER_CASCADE = 25
MAX_CASCADES = 300
REAL_PREFIX_SIZE = 20
REAL_HOUR_HORIZON = 3600
REAL_DECAY_GRID = np.geomspace(1 / 86400, 1 / 600, 18)

In [ ]:
# Запустите эту ячейку после того, как настроите DATA_PATH/CLUSTER_COL/TIME_COL.

real_df = load_dataset(DATA_PATH)
print(f"Исходная таблица: {real_df.shape}")
print("Колонки:", list(real_df.columns))

if CLUSTER_COL not in real_df.columns or TIME_COL not in real_df.columns:
    raise ValueError(
        f"Нужные колонки не найдены. Проверьте CLUSTER_COL={CLUSTER_COL!r} и TIME_COL={TIME_COL!r}."
    )

real_cascades = prepare_cascades(
    df=real_df,
    cluster_col=CLUSTER_COL,
    time_col=TIME_COL,
    min_posts=MIN_POSTS_PER_CASCADE,
    max_cascades=MAX_CASCADES,
)

stats_df = pd.DataFrame([
    {
        "n_cascades": len(real_cascades),
        "avg_posts": np.mean([c["n_posts"] for c in real_cascades]) if real_cascades else 0,
        "median_posts": np.median([c["n_posts"] for c in real_cascades]) if real_cascades else 0,
        "avg_duration_hours": np.mean([c["duration_sec"] for c in real_cascades]) / 3600 if real_cascades else 0,
    }
])

stats_df

In [ ]:
real_eval_df = evaluate_cascades(
    cascades=real_cascades,
    prefix_size=REAL_PREFIX_SIZE,
    decay_grid=REAL_DECAY_GRID,
    hour_horizon=REAL_HOUR_HORIZON,
    n_simulations=300,
    seed=RANDOM_SEED,
)

real_metrics = pd.DataFrame({
    "metric": [
        "n_evaluated_cascades",
        "RMSE(next_hour_count)",
        "RMSE(final_size)",
        "corr(alpha, final_size)",
    ],
    "value": [
        int(len(real_eval_df)),
        safe_rmse(real_eval_df["actual_next_hour"], real_eval_df["pred_next_hour"]),
        safe_rmse(real_eval_df["actual_total"], real_eval_df["pred_total"]),
        safe_corr(real_eval_df["alpha"], real_eval_df["actual_total"]),
    ],
})

real_metrics

In [ ]:
# Визуализация одного реального каскада.
# Берём каскад с максимальным observed final size среди тех, что попали в оценку.

if real_eval_df.empty:
    raise ValueError("После фильтрации и оценки не осталось каскадов. Ослабьте MIN_POSTS_PER_CASCADE или проверьте колонки.")

best_row = real_eval_df.sort_values("actual_total", ascending=False).iloc[0]
example_cluster_id = best_row.get("cluster_id", None)
example_cascade = next(c for c in real_cascades if c["cluster_id"] == example_cluster_id)
example_ts = np.asarray(example_cascade["timestamps"], dtype=float)

# Для красивого графика интенсивности обучаемся на полном каскаде.
full_fit = fit_hawkes_exp_grid(example_ts, REAL_DECAY_GRID)
full_grid = np.linspace(0, example_ts[-1], 700)
full_intensity = intensity_curve(
    mu=full_fit["mu"],
    alpha=full_fit["alpha"],
    decay=full_fit["decay"],
    history=example_ts,
    grid=full_grid,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].step(example_ts / 3600, np.arange(1, len(example_ts) + 1), where="post", color="navy")
axes[0].axvline(example_ts[REAL_PREFIX_SIZE - 1] / 3600, color="red", linestyle="--", label=f"prefix N={REAL_PREFIX_SIZE}")
axes[0].set_title(f"Каскад {example_cluster_id}: накопленное число публикаций")
axes[0].set_xlabel("Время, часы")
axes[0].set_ylabel("Число публикаций")
axes[0].legend()

axes[1].plot(full_grid / 3600, full_intensity, color="darkorange")
axes[1].vlines(example_ts / 3600, ymin=0, ymax=full_intensity.max(), color="gray", alpha=0.15)
axes[1].set_title(f"Каскад {example_cluster_id}: оценённая интенсивность")
axes[1].set_xlabel("Время, часы")
axes[1].set_ylabel("Интенсивность")

plt.tight_layout()
plt.show()

In [ ]:
if real_eval_df.empty:
    raise ValueError("Нет оценённых каскадов для построения scatter-графиков.")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].scatter(real_eval_df["alpha"], real_eval_df["actual_total"], alpha=0.75)
axes[0].set_title("Связь alpha и реального размера каскада")
axes[0].set_xlabel("Оценка alpha по первым N событиям")
axes[0].set_ylabel("Фактический итоговый размер каскада")

axes[1].scatter(real_eval_df["actual_total"], real_eval_df["pred_total"], alpha=0.75)
line_min = min(real_eval_df["actual_total"].min(), real_eval_df["pred_total"].min())
line_max = max(real_eval_df["actual_total"].max(), real_eval_df["pred_total"].max())
axes[1].plot([line_min, line_max], [line_min, line_max], linestyle="--", color="black")
axes[1].set_title("Прогноз итогового размера каскада")
axes[1].set_xlabel("Фактический размер")
axes[1].set_ylabel("Предсказанный размер")

plt.tight_layout()
plt.show()

## Что писать в дипломе по результатам ноутбука

### Для синтетики

Можно формулировать так:

- На синтетически сгенерированных каскадах Hawkes-модель восстанавливает параметры процесса с приемлемой точностью.
- Это подтверждает корректность выбранного подхода и реализации.

### Для реальных данных

Можно формулировать так:

- Для каждого информационного события поток публикаций интерпретируется как самовозбуждающийся точечный процесс.
- Параметр `alpha` характеризует силу каскадного эффекта: чем он выше, тем сильнее одно сообщение стимулирует появление следующих.
- Параметр `mu` отражает фоновую активность, не связанную напрямую с внутренним каскадным размножением.
- Параметр `decay` показывает, как быстро затухает интерес к событию.
- Прогноз по первым `N` сообщениям позволяет оценить дальнейшую динамику обсуждения и ожидаемый размер каскада.

### Ограничения метода

Нормально отдельно указать и ограничения:

- Нужна предварительная кластеризация новостей по событиям.
- Одномерный Hawkes не учитывает тип источника, содержание текста и внешние шоки.
- Оценка итогового размера чувствительна к длине окна наблюдения.
- Для очень коротких каскадов параметры оцениваются нестабильно.